In [26]:
!pip install -U pysr
!pip install plotly

  Obtaining dependency information for pysr from https://files.pythonhosted.org/packages/1a/64/87547c34d4e88aa8c7976fa1be8627ab8a06fccb4d8710224391f6df8a0d/pysr-1.5.10-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: pysr
    Found existing installation: pysr 1.5.9
    Uninstalling pysr-1.5.9:
      Successfully uninstalled pysr-1.5.9


In [16]:
import pysr
import numpy as np
import h5py
import pandas as pd
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sympy import exp, symbols, Function
import sympy as sp

# --- 1. User Definitions ---
bminfit = 3
etamin = 6
etamax = 10

pl_to_zeta = {
    -4: 0.785756,
    -3: 0.539684,
    -2: 0.294367,
    -1: 0.090772
}

sorted_PL_list = sorted(pl_to_zeta.keys())

param_names = [
    'a_re', 'c_re', 'j_re', 'd_re',
    'a_im', 'c_im', 'j_im', 'd_im',
    'a_reA12B', 'c_reA12B', 'j_reA12B', 'd_reA12B'
]

# --- 2. HDF5 Loader Function ---
def load_params_from_h5(filename):
    fitted_params = {}
    with h5py.File(filename, "r") as f:
        param_group = f["jackknife_samples"]
        for key in param_group.keys():
            fitted_params[key] = param_group[key][:]
    return fitted_params

# --- 3. Helper Functions ---
def Jackknife(datalist): 
    N = len(datalist)
    theta_bar = np.mean(datalist)
    theta_nminus_theta_bar = []
    for i in range(N): 
        theta_n = datalist[i]
        theta_nminus_theta_bar.append(np.square(theta_n - theta_bar))
    sigma_sq = ((N-1)/N) * np.sum(theta_nminus_theta_bar)
    return theta_bar, np.sqrt(sigma_sq)

# --- 4. Data Extraction & Scaling ---
summary_data = {key: {'mean': np.zeros(len(sorted_PL_list)), 'error': np.zeros(len(sorted_PL_list))} for key in param_names}

print("Loading and scaling data from HDF5 files...")
for i, pl in enumerate(sorted_PL_list):
    filename = f"/Users/hariprashadravikumar/sivers_TMD_PhD_project/save_h5_A12B_A2B/SimultaniousFit/FitParams_SimulFit_f_A2BRe_cetasq_A2BIm_A12BRe_bmin{bminfit}_eta{etamin}{etamax}_PL{pl}.h5"
    
    params = load_params_from_h5(filename)
    
    scaled_params = {
        'a_re': params['a_re'],
        'c_re': params['c_re'] / (pl**2),
        'j_re': params['j_re'],
        'd_re': params['d_re'],
        
        'a_im': params['a_im'] / pl,
        'c_im': params['c_im'] / (pl**2),
        'j_im': params['j_im'],
        'd_im': params['d_im'],
        
        'a_reA12B': -params['a_reA12B'],
        'c_reA12B': params['c_reA12B'] / (pl**2),
        'j_reA12B': params['j_reA12B'],
        'd_reA12B': params['d_reA12B']
    }
    
    for key in param_names:
        mean, error = Jackknife(scaled_params[key])
        summary_data[key]['mean'][i] = mean
        summary_data[key]['error'][i] = error

print("Data loaded successfully.\n")

# --- 5. PySR Symbolic Regression Setup ---
inv_zeta_list = [1.0 / pl_to_zeta[pl] for pl in sorted_PL_list]
X = np.array(inv_zeta_list).reshape(-1, 1)

# Custom Julia Loss Function 
# (Added a max() safeguard on N-3 to prevent division by zero crashes if your dataset size changes)
custom_loss_function = """
function eval_loss(tree, dataset::Dataset{T,L}, options)::L where {T,L}
    prediction, flag = eval_tree_array(tree, dataset.X, options)
    if !flag
        return L(Inf)
    end
    
    # Just return pure Chi-Squared. 
    # PySR will use its built-in parsimony penalty to punish overly complex equations.
    chi_sq = sum(((prediction .- dataset.y).^2) .* dataset.weights)
    return L(chi_sq)
end
"""



# Initialize PySR Regressor with your custom loss
model = PySRRegressor(
    niterations=200,
    maxsize=50,
    binary_operators=["+", "*"],
    unary_operators=["cosh"],
    model_selection="best",
    loss_function=custom_loss_function,
    #progress=False  # Silences the Jupyter progress bar warnings
    #constraints={'^': (-1, 1)},
    update=True
)

key = "a_im"
y = summary_data[key]['mean']
errors = summary_data[key]['error']
weights = 1.0 / (np.maximum(errors, 1e-10) ** 2)
    
model.fit(X, y, weights=weights, variable_names=["inv_zeta"])


Loading and scaling data from HDF5 files...
Data loaded successfully.



/opt/homebrew/anaconda3/lib/python3.11/site-packages/pysr/sr.py:1469: UserWarning: Note: Using a large maxsize for the equation search will be exponentially slower and use significant memory.
  warnings.warn(
/opt/homebrew/anaconda3/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 4.820e+05
Progress: 1569 / 6200 total iterations (25.306%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.339e+02  0.000e+00  y = -0.10449
3           3.382e+02  1.246e-01  y = inv_zeta * -0.040091
5           1.095e+02  5.638e-01  y = (inv_zeta * -0.02432) + -0.054425
7           8.650e+00  1.269e+00  y = ((inv_zeta * 0.0035751) + -0.063922) * inv_zeta
8           4.594e+00  6.329e-01  y = inv_zeta * (cosh(inv_zeta * 0.02285) + -1.057)
10          4.006e+00  6.847e-02  y = ((cosh(inv_zeta * -0.12269) + -2.8463) * inv_zeta) * 0...
                                      .03095
11          1.609e+00  9.123e-01  y = ((((inv_zeta * -0.25235) + 4.2046) * -0.018381) * inv_...
                                      zeta) + 0.021659
12          4.567e-03

[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	    pick      score                                           equation  \
	0          0.000000                                        -0.10449108   
	1          0.124627                             inv_zeta * -0.04009075   
	2          0.563852                (inv_zeta + 2.2378714) * -0.0243202   
	3          1.269125  ((inv_zeta * 0.0035746791) + -0.063919425) * i...   
	4          0.645840  (cosh(inv_zeta) * 1.1353003e-5) + (inv_zeta * ...   
	5          1.036317  (((inv_zeta * 0.00463829) + -0.07728445) * inv...   
	6          0.000026  (inv_zeta + 0.021657353) + (((inv_zeta * 0.004...   
	7         10.404917  cosh((inv_zeta * -0.17782353) + 2.2186182) * (...   
	8          0.014678  ((inv_zeta + inv_zeta) * (cosh(((((inv_zeta + ...   
	9          0.649896  (-0.014156975 * (((inv_zeta + -0.10132701) + i...   
	10         1.346063  (cosh((((inv_zeta + (((inv_zeta * -0.08534603)...   
	11         0.064090  ((cosh(((((((inv_zeta + -0.28278074) + ((inv_z...   
	12         0.081346  (((inv_zeta + inv_zeta) * cosh(((((((inv_zeta ...   
	13         0.731437  ((cosh((((inv_zeta + (((inv_zeta + (((inv_zeta...   
	14         0.483066  (((inv_zeta + inv_zeta) * cosh(((inv_zeta + ((...   
	15         0.531161  ((cosh((((((((((inv_zeta * inv_zeta) * 0.00621...   
	16  >>>>   1.493485  ((inv_zeta + inv_zeta) * (cosh(((inv_zeta + ((...   
	
	            loss  complexity  
	0   4.338977e+02           1  
	1   3.381719e+02           3  
	2   1.094920e+02           5  
	3   8.650371e+00           7  
	4   4.534716e+00           8  
	5   1.608731e+00           9  
	6   1.608646e+00          11  
	7   4.871496e-05          12  
	8   4.206462e-05          22  
	9   1.146633e-05          24  
	10  7.766924e-07          26  
	11  6.832523e-07          28  
	12  5.806644e-07          30  
	13  1.344643e-07          32  
	14  4.571483e-09          39  
	15  1.580143e-09          41  
	16  7.970250e-11          43  
]

  - outputs/20260504_211642_hZJluG/hall_of_fame.csv


In [18]:
pick =7
model.sympy(pick)

cosh(2.2186182 + inv_zeta*(-0.17782353))*(inv_zeta - 0.54461366)*(-0.02469382)

In [13]:
inv_zeta = np.array([[0.0]])
model.predict(inv_zeta, pick)[0]

0.067904192343651